# Experiment 6 — Developer Automation System using MCP Protocol
### Case Study: ShopEase Pre-Merge Automation (Developer Tools Domain)

**Scenario.** *ShopEase* is a small e-commerce start-up. Before a developer's change to the
checkout module (`cart.py`) is merged, the team wants an AI agent to inspect the code, search it
and run its tests, **without** giving the agent direct access to the file system or shell.
Instead, every developer capability is exposed through a **Model Context Protocol (MCP) server**,
and the agent talks to it through an **MCP client** over **stdio**.

```
Developer request
      │
 Planner Agent ──► Supervisor Agent ──► Safety / Policy layer
                                               │
                                  MCP Client (ClientSession, stdio)
                                               │
                                  MCP Server (FastMCP): list_files · read_file ·
                                  search_code · analyze_python · run_tests
                                               │
        Validator ──► Retry / Recovery ──► Trace (JSONL) ──► Performance analysis ──► Report
```

Run all cells from top to bottom (Runtime → Run all). No API key is needed.

## 1. Install and configure

In [ ]:
# mcp 2.x renamed FastMCP, so we pin the 1.x SDK that this experiment is written against.
%pip install -q "mcp<2" pytest

In [ ]:
import asyncio, json, logging, re, sys, time, uuid
from contextlib import asynccontextmanager
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
from mcp import ClientSession
from mcp.client.stdio import stdio_client, StdioServerParameters

ROOT = Path.cwd()                                  # /content in Google Colab
PROJECT_DIR = ROOT / "sample_developer_project"
LOG_DIR = ROOT / "mcp_logs"
SERVER_FILE = ROOT / "mcp_dev_server.py"
TRACE_FILE = LOG_DIR / "mcp_traces.jsonl"

PROJECT_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)
TRACE_FILE.write_text("")                          # fresh trace file for every run

logging.basicConfig(filename=LOG_DIR / "execution.log", level=logging.INFO, force=True,
                    format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("mcp-dev-automation")
log.info("Experiment 6 started")
print("Folders ready:", PROJECT_DIR.name, "|", LOG_DIR.name)

## 2. Sample developer project (the ShopEase checkout module)

In [ ]:
%%writefile sample_developer_project/cart.py
"""ShopEase checkout helpers."""


def add_item(cart, name, price, qty=1):
    """Add an item to the cart, or increase its quantity."""
    if price < 0 or qty <= 0:
        raise ValueError("price must be >= 0 and qty must be > 0")
    if name in cart:
        cart[name]["qty"] += qty
    else:
        cart[name] = {"price": price, "qty": qty}
    return cart


def remove_item(cart, name):
    """Remove an item from the cart if it is present."""
    cart.pop(name, None)
    return cart


def apply_discount(subtotal, percent):
    """Return the subtotal after a percentage discount."""
    if not 0 <= percent <= 100:
        raise ValueError("percent must be between 0 and 100")
    return round(subtotal * (1 - percent / 100), 2)


def calculate_tax(amount, rate=0.18):
    """GST on an amount (18 percent by default)."""
    return round(amount * rate, 2)


def calculate_total(cart, discount_percent=0):
    """Cart total = (subtotal - discount) + GST."""
    subtotal = sum(i["price"] * i["qty"] for i in cart.values())
    discounted = apply_discount(subtotal, discount_percent)
    return round(discounted + calculate_tax(discounted), 2)

In [ ]:
%%writefile sample_developer_project/app.py
from cart import add_item, calculate_total

cart = {}
add_item(cart, "keyboard", 1500, 1)
add_item(cart, "mouse", 700, 2)
print("Order total with 10 percent coupon:", calculate_total(cart, discount_percent=10))

In [ ]:
%%writefile sample_developer_project/test_cart.py
import pytest
from cart import add_item, remove_item, apply_discount, calculate_tax, calculate_total


def test_add_item_increases_quantity():
    cart = add_item({}, "pen", 10, 2)
    add_item(cart, "pen", 10, 3)
    assert cart["pen"]["qty"] == 5


def test_remove_item():
    cart = add_item({}, "pen", 10)
    assert "pen" not in remove_item(cart, "pen")


def test_apply_discount():
    assert apply_discount(1000, 10) == 900.0


def test_invalid_discount_rejected():
    with pytest.raises(ValueError):
        apply_discount(1000, 150)


def test_calculate_tax():
    assert calculate_tax(1000) == 180.0


def test_calculate_total_with_discount():
    cart = add_item({}, "keyboard", 1000, 1)
    assert calculate_total(cart, 10) == 1062.0

In [ ]:
%%writefile sample_developer_project/README.md
# ShopEase checkout module
Small cart and pricing helpers used for the MCP developer-automation experiment.

## 3. MCP server (FastMCP) — the five developer tools

Each tool is registered with `@mcp.tool()`. File access goes through `safe_project_path()`, which
resolves the path and refuses anything outside the project directory. Tools return a JSON string
with `status` and, on failure, an error `category` that the client uses for recovery decisions.

In [ ]:
%%writefile mcp_dev_server.py
import ast, asyncio, json, os, re, sys, time
from pathlib import Path
from mcp.server.fastmcp import FastMCP

PROJECT_DIR = Path(sys.argv[1]).resolve()
SKIP_DIRS = {"__pycache__", ".pytest_cache"}
mcp = FastMCP("Developer Automation Server")


def ok(**data):
    return json.dumps({"status": "ok", **data})


def fail(category, message):
    return json.dumps({"status": "error", "category": category, "message": message})


def safe_project_path(relative_path):
    """Resolve a path and make sure it stays inside the project directory."""
    candidate = (PROJECT_DIR / relative_path).resolve()
    if candidate != PROJECT_DIR and PROJECT_DIR not in candidate.parents:
        raise PermissionError(f"Path outside project: {relative_path}")
    return candidate


def check_glob(pattern):
    p = Path(pattern)
    if p.is_absolute() or ".." in p.parts:
        raise PermissionError(f"Pattern outside project: {pattern}")


def project_files(pattern):
    check_glob(pattern)
    return sorted(p for p in PROJECT_DIR.rglob(pattern)
                  if p.is_file() and not SKIP_DIRS & set(p.parts))


@mcp.tool()
def list_files(pattern: str = "*") -> str:
    """List project files that match a glob pattern. Risk: LOW."""
    try:
        files = [p.relative_to(PROJECT_DIR).as_posix() for p in project_files(pattern)]
    except PermissionError as e:
        return fail("security_error", str(e))
    return ok(files=files, count=len(files))


@mcp.tool()
def read_file(file_path: str) -> str:
    """Read a text file inside the project. Risk: LOW."""
    try:
        path = safe_project_path(file_path)
    except PermissionError as e:
        return fail("security_error", str(e))
    if not path.is_file():
        return fail("input_error", f"File not found: {file_path}")
    text = path.read_text(encoding="utf-8")
    return ok(file=file_path, content=text, lines=len(text.splitlines()))


@mcp.tool()
def search_code(pattern: str, file_glob: str = "*.py") -> str:
    """Regex search through the project's source files. Risk: LOW."""
    try:
        regex = re.compile(pattern)
        files = project_files(file_glob)
    except re.error as e:
        return fail("input_error", f"Invalid regex: {e}")
    except PermissionError as e:
        return fail("security_error", str(e))
    matches = []
    for path in files:
        for number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
            if regex.search(line):
                matches.append({"file": path.relative_to(PROJECT_DIR).as_posix(),
                                "line": number, "text": line.strip()})
    return ok(pattern=pattern, matches=matches[:200], total=len(matches))


@mcp.tool()
def analyze_python(file_path: str) -> str:
    """AST-based analysis of one Python file. Risk: LOW."""
    try:
        path = safe_project_path(file_path)
    except PermissionError as e:
        return fail("security_error", str(e))
    if not path.is_file() or path.suffix != ".py":
        return fail("input_error", f"Python file not found: {file_path}")
    source = path.read_text(encoding="utf-8")
    try:
        tree = ast.parse(source)
    except SyntaxError as e:
        return fail("input_error", f"Syntax error: {e}")
    functions = [{"name": n.name, "args": [a.arg for a in n.args.args], "line": n.lineno,
                  "has_docstring": ast.get_docstring(n) is not None}
                 for n in ast.walk(tree) if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]
    classes = [n.name for n in ast.walk(tree) if isinstance(n, ast.ClassDef)]
    return ok(file=file_path, lines=len(source.splitlines()),
              functions=sorted(functions, key=lambda f: f["line"]), classes=classes)


@mcp.tool()
async def run_tests(target: str = ".", timeout_seconds: float = 30) -> str:
    """Run pytest in a separate process with a timeout. Risk: MEDIUM."""
    try:
        path = safe_project_path(target)
    except PermissionError as e:
        return fail("security_error", str(e))
    if not path.exists():
        return fail("input_error", f"Test target not found: {target}")
    started = time.perf_counter()
    proc = await asyncio.create_subprocess_exec(
        sys.executable, "-m", "pytest", str(path), "-q", "-p", "no:cacheprovider",
        cwd=str(PROJECT_DIR), env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"},
        stdin=asyncio.subprocess.DEVNULL,        # never share the MCP stdio pipe with the child
        stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.STDOUT)
    try:
        output, _ = await asyncio.wait_for(proc.communicate(), timeout=float(timeout_seconds))
    except asyncio.TimeoutError:
        proc.kill()
        await proc.wait()
        return fail("timeout", f"pytest exceeded {timeout_seconds} s")
    text = output.decode(errors="replace")
    passed = re.search(r"(\d+) passed", text)
    failed = re.search(r"(\d+) failed", text)
    return ok(returncode=proc.returncode, passed=int(passed.group(1)) if passed else 0,
              failed=int(failed.group(1)) if failed else 0,
              seconds=round(time.perf_counter() - started, 2), output=text[-800:])


if __name__ == "__main__":
    mcp.run(transport="stdio")

## 4. MCP client — session, tool discovery and tool invocation

`stdio_client` launches the server as a child process; `ClientSession` speaks MCP over its
stdin/stdout. Tools are **discovered** with `list_tools()` and **invoked** with `call_tool()`.

In [ ]:
@asynccontextmanager
async def mcp_session():
    """Start the MCP server over stdio and yield an initialised ClientSession."""
    params = StdioServerParameters(command=sys.executable, args=[str(SERVER_FILE), str(PROJECT_DIR)])
    with open(LOG_DIR / "mcp_server_stderr.log", "a") as errlog:
        async with stdio_client(params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session


async def call_tool(session, name, arguments=None):
    """Call an MCP tool and decode its JSON result."""
    result = await session.call_tool(name, arguments=arguments or {})
    text = result.content[0].text if result.content else ""
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"status": "error", "category": "execution_error", "message": text[:200] or "empty response"}


RISK_LEVELS = {"list_files": "LOW", "read_file": "LOW", "search_code": "LOW",
               "analyze_python": "LOW", "run_tests": "MEDIUM"}

async with mcp_session() as session:
    AVAILABLE_TOOLS = {t.name: t for t in (await session.list_tools()).tools}

print(f"MCP tool discovery -> {len(AVAILABLE_TOOLS)} tools found via session.list_tools()\n")
print(f"{'Tool':<16}{'Risk':<8}{'Arguments':<34}Description")
for name, tool in AVAILABLE_TOOLS.items():
    props = tool.inputSchema.get("properties", {})
    required = set(tool.inputSchema.get("required", []))
    args = ", ".join(a + ("*" if a in required else "") for a in props)
    print(f"{name:<16}{RISK_LEVELS[name]:<8}{args:<34}{tool.description.split('.')[0]}")
print("\n(* = required argument)")

In [ ]:
def preview(payload, limit=95):
    text = json.dumps(payload)
    return text if len(text) <= limit else text[:limit] + " ..."

demo_calls = [
    ("list_files",     {}),
    ("read_file",      {"file_path": "README.md"}),
    ("search_code",    {"pattern": r"^def \w+"}),
    ("analyze_python", {"file_path": "cart.py"}),
    ("run_tests",      {}),
    ("read_file",      {"file_path": "../../etc/passwd"}),
]
async with mcp_session() as session:
    for name, args in demo_calls:
        reply = await call_tool(session, name, args)
        print(f"call_tool('{name}', {args})")
        print(f"   -> {preview({k: v for k, v in reply.items() if k not in ('content', 'output')})}\n")

## 5. Planner, Supervisor and Safety/Policy layer

* **Planner** — turns the request into structured tasks (with `depends_on`).
* **Supervisor** — approves a task only if the tool exists on the server and the arguments match its schema.
* **Policy** — allows a tool only if its risk level is within the permitted maximum.

In [ ]:
class PlannerAgent:
    """Converts a natural-language developer request into ordered, structured tasks."""
    FILE_RE = re.compile(r"(?:\.\./)+[\w./-]+|[\w./-]+\.(?:py|md|txt)\b")

    def plan(self, request):
        text, tasks = request.lower(), []
        found = self.FILE_RE.search(request)
        target = found.group(0) if found else None

        def add(tool, args, deps=()):
            task_id = f"TASK-{len(tasks) + 1:03d}"
            tasks.append({"task_id": task_id, "tool": tool, "arguments": args, "depends_on": list(deps)})
            return task_id

        if re.search(r"\b(delete|remove|wipe|erase)\b", text):
            add("delete_files", {"pattern": "*"})
        if re.search(r"\blist\b.*\bfiles?\b", text):
            add("list_files", {})
        if target and re.search(r"\b(read|show|open|view)\b", text):
            add("read_file", {"file_path": target})
        analyze_id = add("analyze_python", {"file_path": target}) if target and "analy" in text else None
        if re.search(r"\b(search|find|grep)\b", text):
            pattern = r"^\s*def\s+\w+" if "function" in text else r"^\s*class\s+\w+" if "class" in text else "TODO"
            add("search_code", {"pattern": pattern})
        if re.search(r"\btests?\b", text):
            args = {}
            limit = re.search(r"timeout of ([\d.]+)", text)
            if limit:
                args["timeout_seconds"] = float(limit.group(1))
            add("run_tests", args, deps=[analyze_id] if analyze_id else [])
        return tasks


class SupervisorAgent:
    """Approves a task only if the tool exists and its arguments fit the tool's schema."""
    def __init__(self, tools):
        self.tools = tools

    def review(self, task):
        tool = self.tools.get(task["tool"])
        if tool is None:
            return False, f"tool '{task['tool']}' is not exposed by the MCP server"
        schema = tool.inputSchema
        missing = [a for a in schema.get("required", []) if a not in task["arguments"]]
        unknown = [a for a in task["arguments"] if a not in schema.get("properties", {})]
        if missing or unknown:
            return False, f"invalid arguments (missing={missing}, unknown={unknown})"
        return True, "approved"


class PolicyLayer:
    """Allows a tool only when its risk level does not exceed the permitted maximum."""
    ORDER = {"LOW": 1, "MEDIUM": 2, "HIGH": 3}

    def __init__(self, max_allowed="MEDIUM"):
        self.max_allowed = max_allowed

    def check(self, tool):
        risk = RISK_LEVELS.get(tool, "HIGH")
        return self.ORDER[risk] <= self.ORDER[self.max_allowed], risk


planner, supervisor = PlannerAgent(), SupervisorAgent(AVAILABLE_TOOLS)
print("Planner, Supervisor and Policy layer ready.")

## 6. Execution engine — validation, failure classification, retry, tracing

Every attempt of every task is written to `mcp_traces.jsonl` with the Trace ID, Task ID, tool,
arguments, status, execution time, timestamp and error. Only **timeouts** and **execution errors**
are retried (at most 3 attempts; a timed-out `run_tests` is retried with a 10x larger timeout).
The **dependency-aware scheduler** runs every task whose dependencies are met in parallel with
`asyncio.gather()`, and releases a dependent task only after its dependencies are validated.

In [ ]:
RETRYABLE = {"timeout", "execution_error"}
MAX_ATTEMPTS = 3

def validate(tool, p):
    """Check status, structure and expected output of a tool result."""
    rules = {
        "list_files":     (lambda p: isinstance(p.get("files"), list) and p.get("count", 0) > 0, "no files returned"),
        "read_file":      (lambda p: bool(p.get("content")), "empty file content"),
        "search_code":    (lambda p: isinstance(p.get("matches"), list), "malformed search result"),
        "analyze_python": (lambda p: isinstance(p.get("functions"), list) and p.get("lines", 0) > 0, "malformed analysis"),
        "run_tests":      (lambda p: p.get("returncode") == 0 and p.get("failed", 1) == 0 and p.get("passed", 0) > 0,
                           "pytest did not pass"),
    }
    check, reason = rules[tool]
    return (True, "") if check(p) else (False, reason)


def summarize(tool, p):
    if tool == "list_files":     return f"{p['count']} files listed"
    if tool == "read_file":      return f"{p['lines']} lines read"
    if tool == "search_code":    return f"{p['total']} matches found"
    if tool == "analyze_python": return f"{len(p['functions'])} functions, {len(p['classes'])} classes"
    if tool == "run_tests":      return f"{p['passed']} passed, {p['failed']} failed, return code {p['returncode']}"


def write_trace(record):
    with TRACE_FILE.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")


class WorkflowEngine:
    def __init__(self, session, trace_id):
        self.session, self.trace_id = session, trace_id

    async def run_task(self, task):
        args, attempts, total_ms, first_failure = dict(task["arguments"]), 0, 0.0, None
        while True:
            attempts += 1
            started = time.perf_counter()
            try:
                payload = await call_tool(self.session, task["tool"], args)
            except Exception as exc:                                  # transport / unexpected failure
                payload = {"status": "error", "category": "execution_error", "message": repr(exc)}
            elapsed = (time.perf_counter() - started) * 1000
            total_ms += elapsed
            if payload.get("status") == "ok":
                valid, reason = validate(task["tool"], payload)
                category, message = (None, "") if valid else ("validation_error", reason)
            else:
                valid, category, message = False, payload.get("category", "tool_error"), payload.get("message", "")
            first_failure = first_failure or category
            write_trace({"trace_id": self.trace_id, "task_id": task["task_id"], "tool": task["tool"],
                         "arguments": args, "attempt": attempts, "status": "success" if valid else "failure",
                         "duration_ms": round(elapsed, 2), "timestamp": datetime.now().isoformat(timespec="milliseconds"),
                         "error": message or None})
            log.info("%s %s %s attempt=%d %s", self.trace_id, task["task_id"], task["tool"], attempts,
                     "OK" if valid else f"FAILED ({category}: {message})")
            if valid:
                return {"task_id": task["task_id"], "tool": task["tool"], "arguments": task["arguments"], "success": True,
                        "category": None, "first_failure": first_failure, "attempts": attempts,
                        "duration_ms": total_ms, "summary": summarize(task["tool"], payload)}
            if category in RETRYABLE and attempts < MAX_ATTEMPTS:
                if task["tool"] == "run_tests":
                    args["timeout_seconds"] = args.get("timeout_seconds", 30) * 10
                await asyncio.sleep(0.2 * attempts)
                continue
            return {"task_id": task["task_id"], "tool": task["tool"], "arguments": task["arguments"], "success": False,
                    "category": category, "first_failure": first_failure, "attempts": attempts,
                    "duration_ms": total_ms, "summary": f"{category}: {message}"}

    async def run(self, tasks, rejected_ids=()):
        done = {tid: False for tid in rejected_ids}                 # task_id -> succeeded?
        pending = {t["task_id"]: t for t in tasks}
        phases, records, number = [], [], 0
        wall_start = time.perf_counter()
        while pending:
            ready = [t for t in pending.values() if all(d in done for d in t["depends_on"])] or list(pending.values())
            gated = [t for t in ready if t["depends_on"]]
            runnable = []
            if gated:                                               # dependency validation phase
                number += 1
                for t in ready:
                    blocked = [d for d in t["depends_on"] if not done.get(d, False)]
                    if blocked:
                        done[t["task_id"]] = False
                        records.append({"task_id": t["task_id"], "tool": t["tool"], "arguments": t["arguments"],
                                        "success": False, "category": "dependency_failed", "first_failure": None,
                                        "attempts": 0, "duration_ms": 0.0, "phase": number,
                                        "summary": f"skipped, dependency {blocked[0]} did not succeed"})
                    else:
                        runnable.append(t)
                phases.append({"phase": number, "type": "dependency validation",
                               "tasks": [t["task_id"] for t in gated], "ms": 0.0, "dop": 0})
            else:
                runnable = ready
            for t in ready:
                pending.pop(t["task_id"], None)
            if runnable:
                number += 1
                started = time.perf_counter()
                results = await asyncio.gather(*[self.run_task(t) for t in runnable])
                phases.append({"phase": number, "type": "parallel" if len(runnable) > 1 else "sequential",
                               "tasks": [t["task_id"] for t in runnable],
                               "ms": (time.perf_counter() - started) * 1000, "dop": len(runnable)})
                for r in results:
                    r["phase"] = number
                    done[r["task_id"]] = r["success"]
                    records.append(r)
        return phases, sorted(records, key=lambda r: r["task_id"]), (time.perf_counter() - wall_start) * 1000

print("Execution engine ready.")

In [ ]:
ALL_WORKFLOWS = []

async def handle_request(session, request, policy=None):
    """Planner -> Supervisor -> Policy -> MCP execution -> validation -> trace."""
    policy = policy or PolicyLayer("MEDIUM")
    trace_id = "TRACE-" + uuid.uuid4().hex[:8].upper()
    plan = planner.plan(request)
    approved, rejected = [], []
    for task in plan:
        accepted, reason = supervisor.review(task)
        if not accepted:
            rejected.append({**task, "stage": "Supervisor", "reason": reason})
            continue
        allowed, risk = policy.check(task["tool"])
        if not allowed:
            rejected.append({**task, "stage": "Policy", "reason": f"{risk} risk exceeds allowed level {policy.max_allowed}"})
            continue
        approved.append(task)
    for r in rejected:
        write_trace({"trace_id": trace_id, "task_id": r["task_id"], "tool": r["tool"], "arguments": r["arguments"],
                     "attempt": 0, "status": "rejected", "duration_ms": 0.0,
                     "timestamp": datetime.now().isoformat(timespec="milliseconds"), "error": f"{r['stage']}: {r['reason']}"})
        log.warning("%s %s rejected by %s: %s", trace_id, r["task_id"], r["stage"], r["reason"])
    phases, records, wall_ms = await WorkflowEngine(session, trace_id).run(approved, [r["task_id"] for r in rejected])
    ok_count = sum(r["success"] for r in records)
    wf = {"trace_id": trace_id, "request": request, "plan": plan, "rejected": rejected, "phases": phases,
          "records": records, "wall_ms": wall_ms, "total": len(plan), "ok": ok_count,
          "dop": max([p["dop"] for p in phases if p["type"] == "parallel"], default=1),
          "sequential_tasks": sum(len(p["tasks"]) for p in phases if p["type"] == "sequential"),
          "status": "SUCCESS" if ok_count == len(plan) else "PARTIAL" if ok_count else "FAILED"}
    ALL_WORKFLOWS.append(wf)
    return wf


def fmt_args(a):
    return ", ".join(f"{k}={v}" for k, v in a.items()) or "-"

def show(wf):
    print(f"{wf['trace_id']}  |  {wf['request']}\n\nPLAN")
    for t in wf["plan"]:
        print(f"  {t['task_id']}  {t['tool']:<15}{fmt_args(t['arguments']):<26}depends_on={t['depends_on'] or '-'}")
    print("\nSUPERVISOR / POLICY")
    for t in wf["plan"]:
        rej = next((r for r in wf["rejected"] if r["task_id"] == t["task_id"]), None)
        print(f"  {t['task_id']}  " + (f"REJECTED by {rej['stage']}: {rej['reason']}" if rej
                                      else f"approved (risk {RISK_LEVELS[t['tool']]})"))
    print("\nEXECUTION PHASES")
    for p in wf["phases"]:
        kind = f"{p['type']} (DOP={p['dop']})" if p["type"] == "parallel" else p["type"]
        print(f"  Phase {p['phase']}: {kind:<28}{', '.join(p['tasks']):<22}{p['ms']:7.1f} ms")
    print("\nRESULTS")
    for r in wf["records"]:
        print(f"  {r['task_id']}  {r['tool']:<15}{'OK  ' if r['success'] else 'FAIL'} {r['duration_ms']:7.1f} ms  "
              f"attempts={r['attempts']}  {r['summary']}")
    print(f"\nStatus: {wf['status']}  |  tasks {wf['ok']}/{wf['total']} succeeded  |  DOP={wf['dop']}  |  wall time {wf['wall_ms']:.1f} ms")

## 7. Workflow A — independent tasks run in parallel

In [ ]:
async with mcp_session() as session:
    wf_a = await handle_request(session, "Analyze cart.py and search for all function definitions")
show(wf_a)

## 8. Workflow B — dependency-aware execution (parallel phase, validation, then `run_tests`)

In [ ]:
async with mcp_session() as session:
    wf_b = await handle_request(session, "Analyze cart.py, search for all function definitions and run the tests")
show(wf_b)

## 9. Failure handling, safety and recovery

Five failure scenarios are executed through the same pipeline. The expected behaviour follows the
failure-classification table: input and security errors **stop**, a timeout is **retried**, an unknown
tool is **rejected by the Supervisor**, and a MEDIUM-risk tool is **blocked by a stricter policy**.

In [ ]:
SCENARIOS = [
    ("Missing file",              "Read missing_file.py",                         PolicyLayer("MEDIUM")),
    ("Path traversal",            "Read the file ../../etc/passwd",               PolicyLayer("MEDIUM")),
    ("Unknown tool requested",    "Delete all files in the project",              PolicyLayer("MEDIUM")),
    ("Timeout then retry",        "Run the tests with a timeout of 0.2 seconds",  PolicyLayer("MEDIUM")),
    ("MEDIUM tool under LOW policy", "Analyze cart.py and run its tests",         PolicyLayer("LOW")),
]
SCENARIO_WF, failure_rows = {}, []
async with mcp_session() as session:
    for label, request, policy in SCENARIOS:
        wf = await handle_request(session, request, policy)
        SCENARIO_WF[label] = wf
        if wf["rejected"]:
            r = wf["rejected"][0]
            row = (label, f"{r['stage'].lower()} rejection", 0, f"Blocked by {r['stage']}", "Rejected before execution")
        else:
            rec = next((x for x in wf["records"] if not x["success"] or x["attempts"] > 1), wf["records"][0])
            category = rec["first_failure"] or "none"
            action = {"input_error": "Stop (no retry)", "security_error": "Stop (no retry)", "timeout": "Retry (limited)",
                      "tool_error": "Stop and report", "validation_error": "Stop and report"}.get(category, "-")
            row = (label, category, rec["attempts"], action,
                   "Recovered after retry" if rec["success"] else f"Stopped: {rec['summary'][:44]}")
        failure_rows.append(row)

print(f"{'Scenario':<30}{'Failure category':<22}{'Attempts':<10}{'Action':<26}Outcome")
for row in failure_rows:
    print(f"{row[0]:<30}{row[1]:<22}{row[2]:<10}{row[3]:<26}{row[4]}")

## 10. Performance monitoring — sequential vs parallel (Student Variation 9)

The same four independent calls (`analyze_python`, `search_code`, `list_files`, `read_file`) are
executed one after another and then together with `asyncio.gather()` over the same MCP session.
One un-timed warm-up pass is made first, and the comparison is repeated for 5 runs.

In [ ]:
PERF_CALLS = [("analyze_python", {"file_path": "cart.py"}), ("search_code", {"pattern": r"^\s*def\s+\w+"}),
              ("list_files", {}), ("read_file", {"file_path": "cart.py"})]
RUNS = 5
perf_rows, per_tool = [], {name: [] for name, _ in PERF_CALLS}

async with mcp_session() as session:
    for name, args in PERF_CALLS:                                   # warm-up (not recorded)
        await call_tool(session, name, args)
    for run in range(1, RUNS + 1):
        t0 = time.perf_counter()
        for name, args in PERF_CALLS:
            t = time.perf_counter()
            await call_tool(session, name, args)
            per_tool[name].append((time.perf_counter() - t) * 1000)
        seq_ms = (time.perf_counter() - t0) * 1000
        t0 = time.perf_counter()
        await asyncio.gather(*[call_tool(session, n, a) for n, a in PERF_CALLS])
        par_ms = (time.perf_counter() - t0) * 1000
        perf_rows.append((run, seq_ms, par_ms))

avg_seq = sum(r[1] for r in perf_rows) / RUNS
avg_par = sum(r[2] for r in perf_rows) / RUNS
reduction = (avg_seq - avg_par) / avg_seq * 100

print(f"{'Run':<6}{'Sequential (ms)':<18}{'Parallel (ms)'}")
for run, s, p in perf_rows:
    print(f"{run:<6}{s:<18.2f}{p:.2f}")
print(f"{'Avg':<6}{avg_seq:<18.2f}{avg_par:.2f}")
print(f"\nTasks: {len(PERF_CALLS)} | Runs: {RUNS} | Avg time per tool (sequential): {avg_seq / len(PERF_CALLS):.2f} ms")
print(f"Time reduction with parallel execution: {reduction:.2f} %")

x = range(1, RUNS + 1)
plt.figure(figsize=(6.4, 3.0))
plt.bar([i - 0.2 for i in x], [r[1] for r in perf_rows], width=0.4, label="Sequential", color="#4C72B0")
plt.bar([i + 0.2 for i in x], [r[2] for r in perf_rows], width=0.4, label="Parallel", color="#DD8452")
plt.xlabel("Run"); plt.ylabel("Total time (ms)"); plt.xticks(list(x))
plt.title("Sequential vs parallel MCP execution (4 tasks)"); plt.legend(); plt.tight_layout()
plt.savefig(LOG_DIR / "performance_comparison.png", dpi=150)
plt.show()

## 11. Execution traces and the developer automation report

In [ ]:
def read_traces():
    return [json.loads(line) for line in TRACE_FILE.read_text().splitlines() if line.strip()]

traces = read_traces()
retried = [t for t in traces if t["attempt"] > 1]
print(f"mcp_traces.jsonl holds {len(traces)} trace records "
      f"({len({t['trace_id'] for t in traces})} workflows, {len(retried)} retry attempts)\n")
print(f"{'Trace ID':<15}{'Task':<10}{'Tool':<16}{'Att':<5}{'Status':<10}{'Time (ms)':<11}Error")
for t in traces[-8:]:
    print(f"{t['trace_id']:<15}{t['task_id']:<10}{t['tool']:<16}{t['attempt']:<5}{t['status']:<10}"
          f"{t['duration_ms']:<11.2f}{(t['error'] or '-')[:34]}")

In [ ]:
def phase_of(wf, tool):
    return next((r["phase"] for r in wf["records"] if r["tool"] == tool and r["success"]), None)

checks = {
    "MCP tool discovery found all 5 tools":              len(AVAILABLE_TOOLS) == 5,
    "Independent tasks ran in parallel (DOP = 2)":       wf_b["dop"] == 2,
    "run_tests waited for validated analysis":           (phase_of(wf_b, "run_tests") or 0) > (phase_of(wf_b, "analyze_python") or 99),
    "Path traversal rejected (security_error, no retry)": SCENARIO_WF["Path traversal"]["records"][0]["first_failure"] == "security_error"
                                                         and SCENARIO_WF["Path traversal"]["records"][0]["attempts"] == 1,
    "Missing file stopped without retry":                SCENARIO_WF["Missing file"]["records"][0]["attempts"] == 1,
    "Unknown tool rejected by the Supervisor":           SCENARIO_WF["Unknown tool requested"]["rejected"][0]["stage"] == "Supervisor",
    "MEDIUM-risk tool blocked by LOW policy":            SCENARIO_WF["MEDIUM tool under LOW policy"]["rejected"][0]["stage"] == "Policy",
    "Timeout recovered by controlled retry":             SCENARIO_WF["Timeout then retry"]["records"][0]["success"]
                                                         and SCENARIO_WF["Timeout then retry"]["records"][0]["attempts"] > 1,
}
all_tasks = [r for wf in ALL_WORKFLOWS for r in wf["records"]]
summary = {
    "experiment": "Experiment 6 - Developer Automation System using MCP",
    "case_study": "ShopEase pre-merge automation",
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "final_status": "SUCCESS" if all(checks.values()) else "REVIEW REQUIRED",
    "workflows_run": len(ALL_WORKFLOWS), "tasks_executed": len(all_tasks),
    "tasks_succeeded": sum(r["success"] for r in all_tasks), "tasks_failed": sum(not r["success"] for r in all_tasks),
    "tasks_rejected_before_execution": sum(len(wf["rejected"]) for wf in ALL_WORKFLOWS),
    "retry_attempts": len(retried), "degree_of_parallelism": wf_b["dop"],
    "sequential_phases_tasks": wf_b["sequential_tasks"],
    "avg_sequential_ms": round(avg_seq, 2), "avg_parallel_ms": round(avg_par, 2), "time_reduction_pct": round(reduction, 2),
    "checks": checks,
}
report = {"tools": {n: {"risk": RISK_LEVELS[n], "arguments": list(t.inputSchema.get("properties", {}))} for n, t in AVAILABLE_TOOLS.items()},
          "workflows": ALL_WORKFLOWS, "failure_scenarios": [dict(zip(("scenario", "category", "attempts", "action", "outcome"), r)) for r in failure_rows]}
complete = {**summary, **report, "performance": {"runs": [dict(zip(("run", "sequential_ms", "parallel_ms"), r)) for r in perf_rows],
                                                 "avg_ms_per_tool": {n: round(sum(v) / len(v), 2) for n, v in per_tool.items()}},
            "trace_records": len(traces)}

(LOG_DIR / "developer_automation_report.json").write_text(json.dumps(report, indent=2, default=str))
(LOG_DIR / "experiment_06_complete_report.json").write_text(json.dumps(complete, indent=2, default=str))
(LOG_DIR / "experiment_06_execution_summary.json").write_text(json.dumps(summary, indent=2, default=str))

print("VERIFICATION")
for text, passed in checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {text}")
print(f"\nFinal status: {summary['final_status']}  |  {summary['workflows_run']} workflows, {summary['tasks_executed']} tasks executed: "
      f"{summary['tasks_succeeded']} succeeded, {summary['tasks_failed']} failed (the deliberate failure tests), "
      f"{summary['tasks_rejected_before_execution']} more rejected before execution")
print("\nArtifacts generated in mcp_logs/:")
for f in sorted(LOG_DIR.iterdir()):
    print(f"  {f.name:<42}{f.stat().st_size:>8,} bytes")

## Conclusion

The MCP server exposed five developer tools, the client discovered and invoked them through
`ClientSession`, and the planner, supervisor and policy layers decided what was allowed to run.
Independent tasks ran in parallel (DOP = 2), the dependent `run_tests` task waited for validated
analysis, unsafe requests were rejected, only the timeout was retried, and every step was traced
in `mcp_traces.jsonl`. For such very short tool calls, parallel execution gives only a small gain —
the benefit grows with longer-running tools such as `run_tests`.